$\textbf{1: All imports}$

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import pandas as pd

$\textbf{2: Loading model}$

In [3]:
model_name = "./Models/qwen2.5-transformers-0.5b-instruct-gptq-int4-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModel.from_pretrained(model_name)

$\textbf{3: Creating dataset}$

In [4]:
class RedditGuidelineDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = (
            f"Subreddit: {row['subreddit']}\n"
            f"Rule: {row['rule']}\n"
            f"Positive examples: {row['positive_example_1']} || {row['positive_example_2']}\n"
            f"Negative examples: {row['negative_example_1']} || {row['negative_example_2']}\n"
            f"Comment: {row['body']}"
        )
        label = int(row["rule_violation"])
        return {"text": text, "label": label}

def collate_batch(batch, tokenizer, cap_len=1024, pad_to_multiple_of=8):
    texts = [b["text"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)

    enc = tokenizer(
        texts,
        padding="longest",          # <-- dynamic padding to batch max length
        truncation=True,
        max_length=cap_len,         # safety cap; raise/lower as you like
        return_tensors="pt",
    )

    if pad_to_multiple_of:
        # re-pad to multiple-of-N for Tensor Cores efficiency (optional)
        pad_len = (-enc["input_ids"].shape[1]) % pad_to_multiple_of
        if pad_len:
            pad_id = tokenizer.pad_token_id
            enc["input_ids"] = torch.nn.functional.pad(enc["input_ids"], (0, pad_len), value=pad_id)
            enc["attention_mask"] = torch.nn.functional.pad(enc["attention_mask"], (0, pad_len), value=0)

    enc["labels"] = labels
    return enc

$\textbf{4: Loading the dataset}$

In [ ]:

df = pd.read_csv("./DataFolder/train.csv")

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = RedditGuidelineDataset(train_df)
val_dataset = RedditGuidelineDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda b: collate_batch(b, tokenizer, cap_len=768)
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_batch(b, tokenizer, cap_len=768)
)


$\textbf{full training loop + other}$

In [ ]:
# ---- imports
import math, gc, os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

# -------------------------------
# 1) Data: Dataset returns raw row
# -------------------------------
class RedditGuidelineDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        label = int(row["rule_violation"])
        return {"row": row, "label": label}

# -------------------------------------------------
# 2) Collate with length budgets (body>rule>extras)
# -------------------------------------------------
def collate_batch(batch, tokenizer, cap_len=768, pad_to_multiple_of=8):
    PAD = tokenizer.pad_token_id or tokenizer.eos_token_id
    BOS = getattr(tokenizer, "bos_token_id", None)
    EOS = tokenizer.eos_token_id

    def safe_str(x):
        # robust against None/NaN/odd types
        if x is None:
            return ""
        try:
            if isinstance(x, float) and math.isnan(x):
                return ""
        except Exception:
            pass
        return str(x)

    def encode_row(row_dict):
        body = safe_str(row_dict.get("body", ""))
        rule = "Rule: " + safe_str(row_dict.get("rule", ""))

        extras = (
            "Subreddit: " + safe_str(row_dict.get("subreddit", "")) + "\n"
            "Positive: " + safe_str(row_dict.get("positive_example_1", "")) + " || " + safe_str(row_dict.get("positive_example_2", "")) + "\n"
            "Negative: " + safe_str(row_dict.get("negative_example_1", "")) + " || " + safe_str(row_dict.get("negative_example_2", "")) + "\n"
        )

        # budgets: keep body+rule, let extras shrink
        body_ids = tokenizer(body, truncation=True, max_length=384, add_special_tokens=False)["input_ids"]
        rule_ids = tokenizer(rule, truncation=True, max_length=96, add_special_tokens=False)["input_ids"]

        overhead = (1 if BOS is not None else 0) + (1 if EOS is not None else 0)
        remaining = max(0, cap_len - overhead - len(body_ids) - len(rule_ids))
        extra_ids = tokenizer(extras, truncation=True, max_length=remaining, add_special_tokens=False)["input_ids"]

        ids = []
        if BOS is not None:
            ids.append(BOS)
        ids += body_ids + rule_ids + extra_ids
        if EOS is not None:
            ids.append(EOS)
        return ids

    ids_batch = [encode_row(b["row"]) for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)

    # pad to longest (optionally up to multiple of 8)
    maxlen = max(len(x) for x in ids_batch)
    if pad_to_multiple_of:
        maxlen += (-maxlen) % pad_to_multiple_of

    input_ids = torch.full((len(ids_batch), maxlen), PAD, dtype=torch.long)
    attention_mask = torch.zeros((len(ids_batch), maxlen), dtype=torch.long)
    for i, ids in enumerate(ids_batch):
        L = min(len(ids), maxlen)
        input_ids[i, :L] = torch.tensor(ids[:L], dtype=torch.long)
        attention_mask[i, :L] = 1

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# ---------------------------------------
# 3) Load dataframes + build DataLoaders
# ---------------------------------------
df = pd.read_csv("./DataFolder/train.csv")

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["rule_violation"])

train_dataset = RedditGuidelineDataset(train_df)
val_dataset   = RedditGuidelineDataset(val_df)

# (optional) handle class imbalance with WeightedRandomSampler
label_counts = train_df["rule_violation"].value_counts().to_dict()
weights = train_df["rule_violation"].map(lambda y: 1.0 / label_counts[y]).values
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# tokenizer/model name: adjust to your checkpoint
name = "Qwen/Qwen2.5-0.5B-Instruct"  # or your local path
tok = AutoTokenizer.from_pretrained(name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,            # uses sampler instead of shuffle=True
    collate_fn=lambda b: collate_batch(b, tok, cap_len=768),
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda b: collate_batch(b, tok, cap_len=768),
    pin_memory=True,
)

# ------------------------------------
# 4) Model with proper classification
# ------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForSequenceClassification.from_pretrained(
    name,
    num_labels=2,
    problem_type="single_label_classification",
)
model.config.pad_token_id = tok.pad_token_id
# regularization knobs
model.config.label_smoothing_factor = 0.05  # 0.05–0.1 is typical
# (If supported by the arch:)
if hasattr(model.config, "hidden_dropout_prob"):
    model.config.hidden_dropout_prob = 0.1
if hasattr(model.config, "attention_probs_dropout_prob"):
    model.config.attention_probs_dropout_prob = 0.1

# -------------------------------------------
# 5) Optional: unfreeze last k transformer layers
#    (or swap for LoRA if you prefer)
# -------------------------------------------
def unfreeze_qwen2_cls(model, k_last_layers=4):
    for p in model.parameters():
        p.requires_grad = False

    # Qwen-style backbone location:
    if not hasattr(model, "model") or not hasattr(model.model, "layers"):
        raise AttributeError("Expected model.model.layers for Qwen2; not found.")
    for p in model.model.layers[-k_last_layers:].parameters():
        p.requires_grad = True

    # classification head (usually 'score')
    head_names = ["score", "classifier"]
    found_head = False
    for name_ in head_names:
        if hasattr(model, name_):
            for p in getattr(model, name_).parameters():
                p.requires_grad = True
            found_head = True
            break
    if not found_head:
        raise AttributeError("Could not find classifier head (tried: score, classifier).")

    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"Trainable tensors: {len(trainable)}")
    for n in trainable[:20]:
        print("  ", n)
    if len(trainable) == 0:
        raise RuntimeError("No trainable parameters after unfreeze!")

unfreeze_qwen2_cls(model, k_last_layers=4)

# -----------------------------
# 6) Optimizer / Scheduler etc.
# -----------------------------
epochs         = 10
learning_rate  = 5e-5    # lower than before
weight_decay   = 0.05
grad_accum     = 4
max_grad_norm  = 1.0

use_bf16  = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

steps_per_epoch    = math.ceil(len(train_loader) / grad_accum)
num_training_steps = epochs * steps_per_epoch
num_warmup_steps   = int(0.06 * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = GradScaler(enabled=(amp_dtype == torch.float16))
model.to(device)

# -----------------------------
# 7) Training loop + early stop
# -----------------------------
best_val = float("inf")
patience = 2
stuck = 0
best_state = None

for epoch in range(1, epochs + 1):
    model.train()
    torch.cuda.empty_cache(); gc.collect()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    optimizer_steps = 0

    for step, batch in enumerate(train_loader, start=1):
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with autocast(dtype=amp_dtype):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss / grad_accum

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            should_step = (step % grad_accum == 0) or (step == len(train_loader))
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                optimizer_steps += 1
        else:
            loss.backward()
            should_step = (step % grad_accum == 0) or (step == len(train_loader))
            if should_step:
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                optimizer_steps += 1

        running_loss += loss.detach().item()

    avg_train_loss = running_loss / max(1, len(train_loader))
    print(f"Epoch {epoch} | steps: {optimizer_steps}/{steps_per_epoch} | train loss: {avg_train_loss:.4f}")

    # ---- EVAL ----
    model.eval()
    correct = 0
    total   = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels         = batch["labels"].to(device, non_blocking=True)

            with autocast(dtype=amp_dtype):
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss_sum += out.loss.item()

            preds = out.logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.numel()

    val_acc  = correct / max(1, total)
    val_loss = val_loss_sum / max(1, len(val_loader))
    print(f"Epoch {epoch}: val loss = {val_loss:.4f} | val acc = {val_acc:.4f}")

    # ---- early stopping ----
    if val_loss < best_val:
        best_val = val_loss
        stuck = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stuck += 1
        if stuck > patience:
            print("Early stopping.")
            break

# restore best weights
if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"Loaded best checkpoint with val loss {best_val:.4f}")
